In [1]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel, wilcoxon, shapiro

# ---------------------------
# Dati: accuracy (%) per soggetto S01–S09
# ---------------------------
data = {
    "EEGNet": {
        "None": [77.08, 51.74, 91.67, 54.17, 63.89, 45.83, 83.68, 77.08, 67.01],
        "RDWT": [79.86, 51.74, 92.71, 57.64, 61.11, 49.65, 86.11, 75.00, 77.08],
    },
    "ShallowConvNet": {
        "None": [72.92, 46.88, 83.68, 47.92, 63.19, 40.97, 81.94, 78.82, 75.35],
        "RDWT": [74.31, 48.26, 83.33, 60.42, 56.94, 38.54, 77.78, 79.86, 77.43],
    },
    "MBEEG_SENet": {
        "None": [82.29, 53.82, 92.36, 65.62, 44.79, 57.64, 86.46, 79.51, 71.88],
        "RDWT": [82.64, 54.17, 91.32, 64.24, 67.01, 57.99, 83.33, 79.51, 74.31],
    },
    "EEGTCNet": {
        "None": [63.19, 49.65, 81.25, 50.69, 62.50, 46.18, 80.90, 68.75, 76.04],
        "RDWT": [75.35, 52.78, 85.76, 60.76, 67.36, 49.65, 80.56, 70.14, 76.74],
    },
}

def holm_bonferroni(pvals, alpha=0.05):
    """Ritorna p-value aggiustati (Holm) e boolean significatività."""
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.empty(m, dtype=float)
    maxval = 0.0
    for rank, idx in enumerate(order, start=1):
        adj_p = (m - rank + 1) * pvals[idx]
        maxval = max(maxval, adj_p)
        adj[idx] = maxval  # step-down
    adj = np.minimum(adj, 1.0)
    return adj, adj < alpha

def analyze_pair(model_name, none_vals, rdwt_vals, alpha=0.05):
    a = np.asarray(none_vals, dtype=float)
    b = np.asarray(rdwt_vals, dtype=float)
    if a.shape != b.shape:
        raise ValueError("Le liste devono avere la stessa lunghezza (stessi soggetti).")
    diff = b - a

    # Normalità delle differenze
    sh_w, sh_p = shapiro(diff)

    # Paired t-test (one-sided: RDWT > None)
    t_res = ttest_rel(b, a, alternative="greater")
    # Wilcoxon (one-sided: RDWT > None). Esclude automaticamente i delta=0 (zero_method='wilcox')
    w_res = wilcoxon(b, a, alternative="greater", zero_method="wilcox", correction=False, mode="auto")

    # Effetti: Cohen's d(z) per dati appaiati, rank-biserial per Wilcoxon
    dz = diff.mean() / diff.std(ddof=1)
    # rank-biserial
    n_nonzero = int(np.count_nonzero(diff != 0))
    total_rank_sum = n_nonzero * (n_nonzero + 1) / 2.0 if n_nonzero > 0 else np.nan
    Wplus = w_res.statistic if n_nonzero > 0 else np.nan
    Wminus = (total_rank_sum - Wplus) if n_nonzero > 0 else np.nan
    r_rb = ((Wplus - Wminus) / total_rank_sum) if n_nonzero > 0 else np.nan

    # Test "consigliato": t-test se normale, altrimenti Wilcoxon
    recommended = "t-test" if sh_p >= 0.05 else "Wilcoxon"
    rec_p = t_res.pvalue if recommended == "t-test" else w_res.pvalue
    significant = rec_p < alpha

    out = {
        "model": model_name,
        "mean_None": a.mean(),
        "mean_RDWT": b.mean(),
        "mean_diff": diff.mean(),
        "shapiro_p": sh_p,
        "ttest_t": t_res.statistic,
        "ttest_p_one_sided": t_res.pvalue,
        "wilcoxon_W": w_res.statistic,
        "wilcoxon_p_one_sided": w_res.pvalue,
        "cohen_dz": dz,
        "rank_biserial_r": r_rb,
        "recommended": recommended,
        "recommended_p": rec_p,
        "significant@0.05": significant,
    }
    return out, diff

# Analisi per tutti i modelli
rows, all_rec_p = [], []
for name, vals in data.items():
    res, diff = analyze_pair(name, vals["None"], vals["RDWT"], alpha=0.05)
    rows.append(res)
    all_rec_p.append(res["recommended_p"])

df = pd.DataFrame(rows).set_index("model")

# Aggiustamento multiplo (Holm-Bonferroni) sui p-value del test consigliato
adj_p, sig_adj = holm_bonferroni(np.array(all_rec_p), alpha=0.05)
df["recommended_p_Holm"] = adj_p
df["significant@0.05_Holm"] = sig_adj

# Ordina colonne per leggibilità
cols = [
    "mean_None","mean_RDWT","mean_diff",
    "shapiro_p",
    "ttest_t","ttest_p_one_sided",
    "wilcoxon_W","wilcoxon_p_one_sided",
    "cohen_dz","rank_biserial_r",
    "recommended","recommended_p","recommended_p_Holm","significant@0.05","significant@0.05_Holm"
]
df = df[cols]

# Stampa risultati
pd.set_option("display.precision", 4)
print(df)

# (Facoltativo) stampa anche i delta per soggetto per ogni modello
for name, vals in data.items():
    d = np.asarray(vals["RDWT"]) - np.asarray(vals["None"])
    print(f"\n{name} – diff per soggetto (RDWT - None):")
    for i, val in enumerate(d, start=1):
        print(f" S{i:02d}: {val:+.2f}")


                mean_None  mean_RDWT  mean_diff   shapiro_p  ttest_t  \
model                                                                  
EEGNet            68.0167    70.1000     2.0833  4.0955e-01   1.6431   
ShallowConvNet    65.7411    66.3189     0.5778  1.2108e-01   0.3269   
MBEEG_SENet       70.4856    72.7244     2.2389  5.5433e-05   0.8785   
EEGTCNet          64.3500    68.7889     4.4389  2.3390e-01   3.1802   

                ttest_p_one_sided  wilcoxon_W  wilcoxon_p_one_sided  cohen_dz  \
model                                                                           
EEGNet                     0.0695        29.5                0.0534    0.5477   
ShallowConvNet             0.3761        23.0                0.5000    0.1090   
MBEEG_SENet                0.2026        20.0                0.3896    0.2928   
EEGTCNet                   0.0065        44.0                0.0039    1.0601   

                rank_biserial_r recommended  recommended_p  \
model             

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:198: UserWarning: Sample size too small for normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)


In [4]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

# Dati per soggetto S01–S09 (come in tabella)
data = {
    "EEGNet": {
        "None": [77.08, 51.74, 91.67, 54.17, 63.89, 45.83, 83.68, 77.08, 67.01],
        "RDWT": [79.86, 51.74, 92.71, 57.64, 61.11, 49.65, 86.11, 75.00, 77.08],
    },
    "ShallowConvNet": {
        "None": [72.92, 46.88, 83.68, 47.92, 63.19, 40.97, 81.94, 78.82, 75.35],
        "RDWT": [74.31, 48.26, 83.33, 60.42, 56.94, 38.54, 77.78, 79.86, 77.43],
    },
    "MBEEG_SENet": {
        "None": [82.29, 53.82, 92.36, 65.62, 44.79, 57.64, 86.46, 79.51, 71.88],
        "RDWT": [82.64, 54.17, 91.32, 64.24, 67.01, 57.99, 83.33, 79.51, 74.31],
    },
    "EEGTCNet": {
        "None": [63.19, 49.65, 81.25, 50.69, 62.50, 46.18, 80.90, 68.75, 76.04],
        "RDWT": [75.35, 52.78, 85.76, 60.76, 67.36, 49.65, 80.56, 70.14, 76.74],
    },
}

def holm_bonferroni(pvals):
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.zeros(m, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order, start=1):
        adj_p = (m - rank + 1) * pvals[idx]
        running = max(running, adj_p)  # step-down monotono
        adj[idx] = running
    return np.minimum(adj, 1.0)

def wilcoxon_one_sided(a, b):
    # Paired, one-sided: H1 RDWT > None
    try:
        return wilcoxon(b, a, alternative="greater", zero_method="wilcox", method="exact").pvalue
    except TypeError:
        # SciPy < 1.11 non supporta 'method'
        return wilcoxon(b, a, alternative="greater", zero_method="wilcox").pvalue

rows = []
for model, vals in data.items():
    p = wilcoxon_one_sided(vals["None"], vals["RDWT"])
    rows.append({"Model": model, "p_wilcoxon_one_sided": p})

df = pd.DataFrame(rows)
df["p_Holm"] = holm_bonferroni(df["p_wilcoxon_one_sided"].values)

# Stampa tabellina (usa p_Holm nella tabella o in nota)
pd.set_option("display.precision", 4)
print(df)

# (Facoltativo) righe commento LaTeX da copiare vicino alle righe della tabella
print("\n% LaTeX: p-value Wilcoxon (paired, one-sided; Holm-corrected tra 4 modelli)")
for _, r in df.iterrows():
    print(f"% {r['Model']}: p = {r['p_wilcoxon_one_sided']:.4f}  (Holm {r['p_Holm']:.4f})")


            Model  p_wilcoxon_one_sided  p_Holm
0          EEGNet                0.0534  0.1603
1  ShallowConvNet                0.5000  0.7792
2     MBEEG_SENet                0.3896  0.7792
3        EEGTCNet                0.0039  0.0156

% LaTeX: p-value Wilcoxon (paired, one-sided; Holm-corrected tra 4 modelli)
% EEGNet: p = 0.0534  (Holm 0.1603)
% ShallowConvNet: p = 0.5000  (Holm 0.7792)
% MBEEG_SENet: p = 0.3896  (Holm 0.7792)
% EEGTCNet: p = 0.0039  (Holm 0.0156)


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:198: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:198: UserWarning: Sample size too small for normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)
